# Comparative Specifications — DiD Analysis

> **Prerequisite:** Run `DID_model_modif_11_05.ipynb` first.
> `df`, `df_clean`, `smf`, `np`, `pd`, `plt` must be in memory.

This notebook estimates four model families for two outcomes:
- **`target_engaged`** — binary: firm has a climate target strategy
- **`firm_growth_ord_d`** — binary: firm's revenue grew (vs. flat/decline)

| Section | Specification | Key coefficient |
|---|---|---|
| 1 | Simple OLS | `treat` |
| 2 | Difference-in-Differences | `treat:post` |
| 3 | Triple Difference (DDD) | `treat:post:impl` |
| 4 | DDD — high-dimensional FEs | `treat:post:impl` |
| 5 | Comparison table | all specs side-by-side |

## Setup

In [ ]:
from statsmodels.iolib.summary2 import summary_col

# DDD sample: impl=0 (never-treated) + impl=1 (policy countries)
df_ddd = df.dropna(subset=[
    "target_engaged", "treat", "post",
    "scr12", "nace_b", "ipscntry", "implementation_country"
]).copy()
df_ddd["impl"] = df_ddd["implementation_country"]

# Shared analysis samples
df_s = df_clean.dropna(subset=["firm_growth_ord_d"]).copy()  # OLS/DiD (impl=1)
df_d = df_ddd.dropna(subset=["firm_growth_ord_d"]).copy()    # DDD (all countries)

# Extended FE dummies for Section 4
df_d["cntry_treat"] = df_d["ipscntry"].astype(str) + "_" + df_d["treat"].astype(str)
df_d["cntry_post"]  = df_d["ipscntry"].astype(str) + "_" + df_d["post"].astype(str)

print(f"OLS/DiD sample (impl=1): {df_s.shape[0]:,} obs")
print(f"DDD sample (all):        {df_d.shape[0]:,} obs")
print(f"  impl=1 (policy countries):  {(df_d['impl']==1).sum():,}")
print(f"  impl=0 (never-treated):     {(df_d['impl']==0).sum():,}")

def cl(df_):
    return {"cov_type": "cluster", "cov_kwds": {"groups": df_["ipscntry"]}}

## 1. Simple OLS
Cross-sectional estimate of the firm-size effect — no pre/post variation exploited.  
`treat = 1` if firm has ≥ 250 employees **or** turnover ≥ €10 M.

| Model | Outcome | Country FE |
|---|---|---|
| 1a | target_engaged | No |
| 1b | target_engaged | Yes (`C(ipscntry)`) |
| 1c | firm_growth_ord_d | No |
| 1d | firm_growth_ord_d | Yes (`C(ipscntry)`) |

In [ ]:
s1a = smf.ols("target_engaged    ~ treat + C(nace_b) + C(firm_age_ord)",                data=df_s).fit(**cl(df_s))
s1b = smf.ols("target_engaged    ~ treat + C(nace_b) + C(firm_age_ord) + C(ipscntry)", data=df_s).fit(**cl(df_s))
s1c = smf.ols("firm_growth_ord_d ~ treat + C(nace_b) + C(firm_age_ord)",                data=df_s).fit(**cl(df_s))
s1d = smf.ols("firm_growth_ord_d ~ treat + C(nace_b) + C(firm_age_ord) + C(ipscntry)", data=df_s).fit(**cl(df_s))

for lbl, m in [("1a  target_engaged    no FE",     s1a),
               ("1b  target_engaged    country FE", s1b),
               ("1c  firm_growth_ord_d no FE",     s1c),
               ("1d  firm_growth_ord_d country FE", s1d)]:
    print(f"{lbl}: treat = {m.params['treat']:.4f}  (p={m.pvalues['treat']:.3f})")

## 2. Difference-in-Differences (DiD)
`treat:post` is the DiD coefficient — the additional change for large firms
after the policy (`post=1`, year 2024) relative to small firms.  
Sample restricted to implementing countries (`impl=1`).

| Model | Outcome | Country FE |
|---|---|---|
| 2a | target_engaged | No |
| 2b | target_engaged | Yes |
| 2c | firm_growth_ord_d | No |
| 2d | firm_growth_ord_d | Yes |

In [ ]:
s2a = smf.ols("target_engaged    ~ treat*post + C(nace_b) + C(firm_age_ord)",                data=df_s).fit(**cl(df_s))
s2b = smf.ols("target_engaged    ~ treat*post + C(nace_b) + C(firm_age_ord) + C(ipscntry)", data=df_s).fit(**cl(df_s))
s2c = smf.ols("firm_growth_ord_d ~ treat*post + C(nace_b) + C(firm_age_ord)",                data=df_s).fit(**cl(df_s))
s2d = smf.ols("firm_growth_ord_d ~ treat*post + C(nace_b) + C(firm_age_ord) + C(ipscntry)", data=df_s).fit(**cl(df_s))

for lbl, m in [("2a  target_engaged    no FE",     s2a),
               ("2b  target_engaged    country FE", s2b),
               ("2c  firm_growth_ord_d no FE",     s2c),
               ("2d  firm_growth_ord_d country FE", s2d)]:
    print(f"{lbl}: treat:post = {m.params['treat:post']:.4f}  (p={m.pvalues['treat:post']:.3f})")

## 3. Difference-in-Difference-in-Differences (DDD)
Adds a third dimension: `impl` (= `implementation_country`).  
Countries with `impl=0` are **never-treated controls**, removing
global time trends that affect large firms everywhere.

`treat:post:impl` is the DDD coefficient.

> **Note:** `C(ipscntry)` is excluded — `impl` is constant within countries
> and would be perfectly collinear with country FEs. Sector and age FEs are
> used for the 'with FE' variant.

| Model | Outcome | Controls |
|---|---|---|
| 3a | target_engaged | Sector only |
| 3b | target_engaged | Sector + firm age |
| 3c | firm_growth_ord_d | Sector only |
| 3d | firm_growth_ord_d | Sector + firm age |

In [ ]:
s3a = smf.ols("target_engaged    ~ treat*post*impl + C(nace_b)",                   data=df_d).fit(**cl(df_d))
s3b = smf.ols("target_engaged    ~ treat*post*impl + C(nace_b) + C(firm_age_ord)", data=df_d).fit(**cl(df_d))
s3c = smf.ols("firm_growth_ord_d ~ treat*post*impl + C(nace_b)",                   data=df_d).fit(**cl(df_d))
s3d = smf.ols("firm_growth_ord_d ~ treat*post*impl + C(nace_b) + C(firm_age_ord)", data=df_d).fit(**cl(df_d))

for lbl, m in [("3a  target_engaged    sector FE",     s3a),
               ("3b  target_engaged    sector+age FE", s3b),
               ("3c  firm_growth_ord_d sector FE",     s3c),
               ("3d  firm_growth_ord_d sector+age FE", s3d)]:
    print(f"{lbl}: treat:post:impl = {m.params['treat:post:impl']:.4f}  "
          f"(p={m.pvalues['treat:post:impl']:.3f})")

## 4. DDD — High-Dimensional Fixed Effects
Extends Section 3 by absorbing all two-way interactions via dedicated FEs:

$$
Y_{ict} = \\beta(Post_t \\times TreatedCountry_c \\times Eligible_i)
         + \\mu_{c \\times Eligible_i}
         + \\lambda_{c \\times t}
         + \\delta_{Eligible_i \\times t}
         + X_{ic} + \\varepsilon_{ict}
$$

| Term | Implementation | Absorbs |
|---|---|---|
| $\\mu_{c \\times Eligible_i}$ | `C(cntry_treat)` | country FE + treat FE + country×treat |
| $\\lambda_{c \\times t}$ | `C(cntry_post)` | country FE + post FE + country×time |
| $\\delta_{Eligible_i \\times t}$ | `treat:post` | eligibility×time (DiD term) |

> **Warning:** Standard errors may be `nan` for `treat:post:impl` if `impl`
> (a country-group constant) is partially collinear with `C(cntry_treat)`
> or `C(cntry_post)`. The coefficient is directionally informative but
> inference requires caution.

| Model | Outcome |
|---|---|
| 4a | target_engaged |
| 4b | firm_growth_ord_d |

In [ ]:
s4a = smf.ols(
    """target_engaged ~ treat:post:impl
       + C(cntry_treat) + C(cntry_post) + treat:post
       + C(nace_b) + C(firm_age_ord)""",
    data=df_d
).fit(**cl(df_d))

s4b = smf.ols(
    """firm_growth_ord_d ~ treat:post:impl
       + C(cntry_treat) + C(cntry_post) + treat:post
       + C(nace_b) + C(firm_age_ord)""",
    data=df_d
).fit(**cl(df_d))

for lbl, m in [("4a  target_engaged", s4a), ("4b  firm_growth_ord_d", s4b)]:
    coef = m.params['treat:post:impl']
    se   = m.bse['treat:post:impl']
    pval = m.pvalues['treat:post:impl']
    print(f"{lbl}: treat:post:impl = {coef:.4f}  SE={se:.4f}  (p={pval:.3f})")

## 5. Comparison Table
All 14 models side-by-side. Only the key treatment coefficient for each
specification is shown. Significance: \* p<0.1, \*\* p<0.05, \*\*\* p<0.01.

In [ ]:
comp_table = summary_col(
    [s1a, s1b, s1c, s1d, s2a, s2b, s2c, s2d, s3a, s3b, s3c, s3d, s4a, s4b],
    model_names=[
        "(1a)","(1b)","(1c)","(1d)",
        "(2a)","(2b)","(2c)","(2d)",
        "(3a)","(3b)","(3c)","(3d)",
        "(4a)","(4b)",
    ],
    stars=True,
    float_format="%0.4f",
    regressor_order=["treat", "treat:post", "treat:post:impl"],
    drop_omitted=True,
    info_dict={
        "N":          lambda x: f"{int(x.nobs)}",
        "R-squared":  lambda x: f"{x.rsquared:.3f}",
        "Outcome":    lambda x: x.model.endog_names,
        "Country FE": lambda x: "Yes" if any("ipscntry" in v for v in x.model.exog_names) else "No",
    },
)
print(comp_table)

In [ ]:
with open("comparison_table.txt", "w") as f:
    f.write(str(comp_table))
print("Saved: comparison_table.txt")